In [1]:
# ---- Imports ----
import kagglehub #install this before running the notebook to download the dataset locally
import os
import re
import ast
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import torch.nn as nn
from tqdm import tqdm
from collections import Counter
import matplotlib.pyplot as plt
import torch.nn.functional as F
import matplotlib.gridspec as gridspec
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModel, AutoModelForCausalLM

## Data exploration

#### Load the CommonsenseQA dataset

In [2]:
# Download dataset using kagglehub
path = kagglehub.dataset_download("thedevastator/new-commonsenseqa-dataset-for-multiple-choice-qu")
print("Path:", path)

# List files inside the dataset directory & display train df 
print(f"Files available: {os.listdir(path)}\n")
df = pd.read_csv(f"{path}/train.csv")
df

# Validation --> Use as test 
df_val = pd.read_csv(f"{path}/validation.csv", index_col=0)
df_val.index.name = "answerKey"
df_val = df_val.reset_index()
print(f"Empty rows Val: {df_val['answerKey'].isnull().sum()}\n") #checked: is 0 
print(f"Quick check: Val split distribution\n{df_val['answerKey'].value_counts().to_dict()}\n")
#df_val

# Test 
df_test = pd.read_csv(f"{path}/test.csv", index_col=0)
df_test.index.name = "answerKey"
df_test = df_test.reset_index()
print(f"Empty rows Test: {df_test['answerKey'].isnull().sum()}") #checked: no rows contain answers (1140)
#df_test


Path: C:\Users\Nour\.cache\kagglehub\datasets\thedevastator\new-commonsenseqa-dataset-for-multiple-choice-qu\versions\2
Files available: ['test.csv', 'train.csv', 'validation.csv']

Empty rows Val: 0

Quick check: Val split distribution
{'B': 255, 'D': 251, 'C': 241, 'A': 239, 'E': 235}

Empty rows Test: 1140


#### Basic exploration 

In [3]:
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nFirst row:\n{df.iloc[0]}")

# Check for missing values 
print("\nMissing values")
print(df.isnull().sum())
print(f"Duplicate rows: {df.duplicated().sum()}")

Shape: (9741, 3)
Columns: ['answerKey', 'question', 'choices']

Dtypes:
answerKey    str
question     str
choices      str
dtype: object

First row:
answerKey                                                    A
question     The sanctions against the school were a punish...
choices      {'label': array(['A', 'B', 'C', 'D', 'E'], dty...
Name: 0, dtype: str

Missing values
answerKey    0
question     0
choices      0
dtype: int64
Duplicate rows: 0


#### Cleaning/Formatting

In [4]:
# Convert the raw string into a real Python object
print(f"Raw string (before applying parse_choices):\n{repr(df['choices'].iloc[0])}\n")

def parse_choices(raw):
    """
    Converts the stringified dict to a plain {label: text} dict
    """
    # Remove numpy array wrapper: array([...], dtype=object) ->  [...]
    cleaned = re.sub(r"array\((\[.*?\]),\s*dtype=object\)", r"\1", raw, flags=re.DOTALL)
    try:
        d = ast.literal_eval(cleaned) # So this string: {'label': array([...]), 'text': array([...])} becomes an actual dict with 2 numpy arrays inside
        return dict(zip(list(d["label"]), list(d["text"]))) #takes the 2 arrays (labels & texts), and pair them up into a clean, usable dict
    except Exception:
        return {}
 
df["choices_parsed"] = df["choices"].apply(parse_choices) #{'A': 'option1', 'B': 'option2', 'C': 'option3', ...}
df["num_choices"] = df["choices_parsed"].apply(len)
print(f"\nChoice counts per question:\n{df['num_choices'].value_counts()}")

# Flatten all choice texts across all rows into one flat list for length analysis
all_choice_texts = [text for c in df["choices_parsed"] for text in c.values()]

# Example of parse_choices applied 
sample = df["choices"].iloc[0]
print(f"\nResulting string (after applying parse_choices):\n{parse_choices(sample)}")


Raw string (before applying parse_choices):
"{'label': array(['A', 'B', 'C', 'D', 'E'], dtype=object), 'text': array(['ignore', 'enforce', 'authoritarian', 'yell at', 'avoid'],\n      dtype=object)}"


Choice counts per question:
num_choices
5    9741
Name: count, dtype: int64

Resulting string (after applying parse_choices):
{'A': 'ignore', 'B': 'enforce', 'C': 'authoritarian', 'D': 'yell at', 'E': 'avoid'}


In [5]:
# Cleaning checks 
print("\nCleaning checklist")
unparseable = (df["num_choices"] == 0).sum()
bad_keys = df.apply(lambda r: r["answerKey"] not in r["choices_parsed"], axis=1).sum()
no_question_mark = (~df["question"].str.strip().str.endswith("?")).sum()
 
print(f" Rows with unparseable choices: {unparseable}") 
print(f" Rows where answerKey is not in choices: {bad_keys}") #all correct answers are present in options 
print(f" Questions missing trailing '?': {no_question_mark}") #not all questions have a question mark 
# Optional: drop bad rows
# df = df[df["num_choices"] > 0].reset_index(drop=True)


Cleaning checklist
 Rows with unparseable choices: 0
 Rows where answerKey is not in choices: 0
 Questions missing trailing '?': 67


#### Feature engineering & Basic statistics 

In [6]:
# Feature Engineering: question level features 
df["q_starts_with"] = df["question"].str.split().str[0].str.lower()
df["q_len_chars"] = df["question"].str.len()
df["q_len_words"] = df["question"].str.split().str.len()
# Mean choice length: characters 
df["choice_len_mean"] = df["choices_parsed"].apply(
     lambda d: np.mean([len(t) for t in d.values()]) if d else np.nan)
# Mean choice length: words
df["choice_len_mean_words"] = df["choices_parsed"].apply(
    lambda d: np.mean([len(t.split()) for t in d.values()]) if d else np.nan)

# Basic statistics: q&a distributions 
print("\nAnswer key distribution")
print("-"*30)
order = sorted(df["answerKey"].dropna().unique())
counts = df["answerKey"].value_counts().reindex(order)
percentages = counts / counts.sum() * 100
summary = pd.DataFrame({
    "Count": counts,
    "Percentage": percentages.round(2)})
print(summary)

print("\nTop 10 question-opening words")
print("-"*30)
print(df["q_starts_with"].value_counts().head(10))

print("\nQuestion length (words)")
print("-"*30)
print(df["q_len_words"].describe())

print("\nQuestion length (characters)")
print("-"*30)
print(df["q_len_chars"].describe())

print("\nMean choice length (characters)")
print("-"*30)
print(df["choice_len_mean"].describe())

print("\nMean choice length (words)")
print("-"*30)
print(df["choice_len_mean_words"].describe())


Answer key distribution
------------------------------
           Count  Percentage
answerKey                   
A           1909       19.60
B           1973       20.25
C           1946       19.98
D           1985       20.38
E           1928       19.79

Top 10 question-opening words
------------------------------
q_starts_with
what     2113
where    1820
the      1267
if        649
when      285
a         278
he        261
john      199
why       186
james     180
Name: count, dtype: int64

Question length (words)
------------------------------
count    9741.000000
mean       13.249974
std         5.516564
min         3.000000
25%         9.000000
50%        12.000000
75%        16.000000
max        63.000000
Name: q_len_words, dtype: float64

Question length (characters)
------------------------------
count    9741.000000
mean       69.466893
std        29.463106
min        15.000000
25%        48.000000
50%        64.000000
75%        84.000000
max       376.000000
Name: q_len_

#### Linguistic feature analysis 
These features proxy for question difficulty and potential model failure modes (factors likely to influence where confidence and correctness diverge).


- Question difficulty signal, proxied with negation: Questions with negation words require more reasoning 
steps and models are known to struggle with them. A predefined 
[negations word list](https://aclanthology.org/P17-1154.pdf) was used for detection.

- Lexical overlap between questions and choices: If a choice reuses words from the question, a model might pick it due to surface-level matching rather than reasoning --> would show up as high confidence on a wrong answer (shortcut behaviour).

- Distractor quality (choice length variation): If all choices are very similar, the task is harder and confidence scores should be lower. If one choice is obviously different, confidence might be high but for the wrong reason

In [7]:
# Flag questions containing negations (models often struggle with these)
df["has_negation"] = df["question"].str.extract(
    r"\b(no|not|never|neither|except|nobody|nothing|nowhere|seldom|scarcely|hardly|barely|cannot|least|without|n't)\b", 
    regex=True, case=False)
#print(df["has_negation"].value_counts()) #counts 
# Convert to percentages for easy interpretation 
negation_pct = df["has_negation"].value_counts(normalize=True) * 100
print(f"Contains negation: {negation_pct[True]:.1f}%  ({df['has_negation'].sum()} questions)")
print(f"No negation: {negation_pct[False]:.1f}%  ({(~df['has_negation']).sum()} questions)")
print("-"*30)


def tokenize(text):
    return set(re.findall(r"\b\w+\b", text.lower()))

def overlap(row):
    """
    Compute word overlap between the question and each answer choice.
    Returns: dict: mapping choice key -> number of shared tokens with the question
    """
    q_words = tokenize(row["question"])
    return {k: len(q_words & set(v.lower().split())) for k, v in row["choices_parsed"].items()}

# Compute overlap scores for each choice per question
df["q_choice_overlap"] = df.apply(overlap, axis=1)
# Extract overlap score for the correct answer
df["correct_overlap"] = df.apply(lambda r: r["q_choice_overlap"].get(r["answerKey"], 0), axis=1)
# Compute maximum overlap among all answer choices
df["max_overlap"] = df["q_choice_overlap"].apply(lambda d: max(d.values()))
# Check if correct answer has the highest overlap
df["correct_is_max_overlap"] = ((df["correct_overlap"] == df["max_overlap"]) &  
    (df["max_overlap"] > 0))

overlap_pct = df["correct_is_max_overlap"].mean() * 100
# If the correct answer consistently has the highest overlap, models may be use surface-level
# word matching rather than reasoning 
print(f"Correct answer has highest word overlap with question in: {overlap_pct:.1f}% of cases")
print(f"{'-> Potential shortcut risk' if overlap_pct > 50 else '-> Limited evidence of overlap bias'}")
print("-"*30)


# Measure how much answer choices vary in length within each question.
df["choice_text_lengths"] = df["choices_parsed"].apply(
    lambda d: [len(v) for v in d.values()])
# Compute standard deviation of choice lengths (per question)
df["choice_len_std"] = df["choice_text_lengths"].apply(np.std)
print(f"Mean std across questions: {df['choice_len_std'].mean():.2f} chars")
print(f"Median std: {df['choice_len_std'].median():.2f} chars")
print(f"Questions with high variance (std > 10): {(df['choice_len_std'] > 10).sum()}")

# High variance (std) suggests uneven distractors (varying a lot in length): one choice may stand out visually,
# which can artificially increase model confidence regardless of correctness

TypeError: StringMethods.extract() got an unexpected keyword argument 'regex'

#### EDA Visualisations 

In [ ]:
# Set up figure
fig = plt.figure(figsize=(16, 12))
fig.suptitle("CommonsenseQA dataset: EDA Overview", fontsize=16, fontweight="bold")
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35) #initialize a 2x3 grid layout for the subplots
 
# Plot A:  Answer key distribution 
ax1 = fig.add_subplot(gs[0, 0])
counts = df["answerKey"].value_counts().reindex(order)
bars = ax1.bar(order, counts.values, color=sns.color_palette("muted", len(order)))
ax1.bar_label(bars, padding=3, fontsize=9)
ax1.set_title("Answer Key Distribution")
ax1.set_xlabel("Choice"); ax1.set_ylabel("Count")
 
# Plot B: Question text length histogram
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(df["q_len_words"].dropna(), bins=30, color="dimgrey", edgecolor="white")
ax2.axvline(df["q_len_words"].median(), color="red", linestyle="--",
            label=f"Median {df['q_len_words'].median():.0f} words")
ax2.set_title("Question Length (words)")
ax2.set_xlabel("Word count"); ax2.set_ylabel("Frequency")
ax2.legend(fontsize=9)
 
# Plot C: Choice text length histogram 
ax3 = fig.add_subplot(gs[0, 2])
choice_lens = [len(t) for t in all_choice_texts]
ax3.hist(choice_lens, bins=40, color="dimgrey", edgecolor="white") 
ax3.axvline(np.median(choice_lens), color="red", linestyle="--",
            label=f"Median {np.median(choice_lens):.0f} chars")
ax3.set_title("Choice Text Length (chars)")
ax3.set_xlabel("Characters"); ax3.set_ylabel("Frequency")
ax3.legend(fontsize=9)
 
# Plot D: Top 10 question-opening words
ax4 = fig.add_subplot(gs[1, 0:2]) #span two columns instead of one 
top_starts = df["q_starts_with"].value_counts().head(10)
ax4.barh(top_starts.index[::-1], top_starts.values[::-1], color="dimgrey")
ax4.set_title("Top 10 Question-Opening Words")
ax4.set_xlabel("Count")
for i, v in enumerate(top_starts.values[::-1]):
    ax4.text(v + 0.5, i, str(v), va="center", fontsize=9)
 
# Plot E: Mean choice length by answer key
ax5 = fig.add_subplot(gs[1, 2])
groups = [df[df["answerKey"] == k]["choice_len_mean"].dropna().values for k in order]
bp = ax5.boxplot(groups, labels=order, patch_artist=True,
                 medianprops=dict(color="black", linewidth=2))
for patch, color in zip(bp["boxes"], sns.color_palette("muted", len(order))):
    patch.set_facecolor(color)
ax5.set_title("Mean Choice Length by Answer Key")
ax5.set_xlabel("Answer Key"); ax5.set_ylabel("Mean chars")
 
plt.tight_layout()
plt.savefig("eda_overview.png", dpi=150, bbox_inches="tight")
print("\nPlot saved -> eda_overview.png")
plt.show()
 

#### Worked examples 
(to get an intuition of what the model will see)

In [ ]:
examples = df.sample(3, random_state=42)

for i, row in examples.iterrows():
    print(f"\n--- Example {i} ---")
    print("Question:", row["question"])
    print("Choices:", row["choices_parsed"])
    print("Answer:", row["answerKey"])

## Model 1: Decoder model GPT-small

In [8]:
# Decoder configuration
decoder_model_name = "openai-community/gpt2"
SEED = 42
SAMPLE_SIZE = 120  # change for testing

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def format_decoder_split(raw_df):
    records = []
    for _, row in raw_df.iterrows():
        choice_map = parse_choices(row["choices"])
        labels = list(choice_map.keys())
        options = list(choice_map.values())
        records.append(
            {
                "question": row["question"],
                "answer_key": row["answerKey"],
                "labels": labels,
                "options": options,
                "choice_map": choice_map,
            }
        )
    return pd.DataFrame(records)

decoder_datasets = {
    "train": format_decoder_split(df),
    "validation": format_decoder_split(df_val),
}
for split_name, split_df in decoder_datasets.items():
    print(f"decoder {split_name}: {len(split_df)} examples")


# Decoder model loading
decoder_tokenizer = AutoTokenizer.from_pretrained(decoder_model_name)
if decoder_tokenizer.pad_token is None:
    decoder_tokenizer.pad_token = decoder_tokenizer.eos_token

decoder_model = AutoModelForCausalLM.from_pretrained(decoder_model_name)
decoder_model.to(DEVICE)
decoder_model.eval()

def build_decoder_prompt(question, labels, options):
    options_block = "\n".join(f"{label}. {text}" for label, text in zip(labels, options))
    return f"Question: {question}\n{options_block}\nAnswer:"

@torch.no_grad()
def score_decoder_option_logprob(prompt, option_text):
    prompt_ids = decoder_tokenizer(prompt, return_tensors="pt", add_special_tokens=False).input_ids.to(DEVICE)
    target_ids = decoder_tokenizer(" " + option_text, return_tensors="pt", add_special_tokens=False).input_ids.to(DEVICE)

    full_ids = torch.cat([prompt_ids, target_ids], dim=1)
    logits = decoder_model(full_ids).logits
    log_probs = torch.log_softmax(logits, dim=-1)

    start = prompt_ids.shape[1]
    total = 0.0
    for pos in range(start, full_ids.shape[1]):
        token_id = full_ids[0, pos]
        total += log_probs[0, pos - 1, token_id].item()
    token_count = max(1, target_ids.shape[1])
    return total / token_count

def predict_decoder_with_confidence(question, labels, options):
    prompt = build_decoder_prompt(question, labels, options)
    option_logps = np.array([score_decoder_option_logprob(prompt, option) for option in options], dtype=np.float64)

    shifted = option_logps - np.max(option_logps)
    probs = np.exp(shifted)
    probs = probs / probs.sum()

    best_idx = int(np.argmax(probs))
    return {
        "pred_label": labels[best_idx],
        "confidence": float(probs[best_idx]),
        "option_probs": {label: float(prob) for label, prob in zip(labels, probs)},
        "option_logprobs": {label: float(lp) for label, lp in zip(labels, option_logps)},
    }

def evaluate_decoder_split(decoder_df, sample_size=None):
    run_df = decoder_df.sample(n=sample_size, random_state=SEED).reset_index(drop=True) if sample_size else decoder_df.reset_index(drop=True)
    rows = []
    for _, row in tqdm(run_df.iterrows(), total=len(run_df), desc="Decoder evaluating"):
        result = predict_decoder_with_confidence(row["question"], row["labels"], row["options"])
        correct = int(result["pred_label"] == row["answer_key"])
        rows.append(
            {
                "question": row["question"],
                "answer_key": row["answer_key"],
                "predicted": result["pred_label"],
                "correct": correct,
                "confidence": result["confidence"],
                "all_probs": result["option_probs"],
            }
        )
    return pd.DataFrame(rows)

def expected_calibration_error(conf, corr, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        left, right = bins[i], bins[i + 1]
        mask = (conf >= left) & (conf < right if i < n_bins - 1 else conf <= right)
        if mask.sum() == 0:
            continue
        bin_acc = corr[mask].mean()
        bin_conf = conf[mask].mean()
        ece += (mask.sum() / len(conf)) * abs(bin_acc - bin_conf)
    return float(ece)

def summarize_decoder_results(results):
    accuracy = float(results["correct"].mean())
    avg_conf = float(results["confidence"].mean())
    corr = float(np.corrcoef(results["confidence"], results["correct"])[0, 1]) if len(results) > 1 else float("nan")
    ece = expected_calibration_error(results["confidence"].to_numpy(), results["correct"].to_numpy(), n_bins=10)
    return {
        "n": int(len(results)),
        "accuracy": accuracy,
        "avg_confidence": avg_conf,
        "confidence_correctness_corr": corr,
        "ece": ece,
    }

# test on validation split
decoder_validation_results = evaluate_decoder_split(decoder_datasets["validation"], sample_size=SAMPLE_SIZE)
decoder_metrics = summarize_decoder_results(decoder_validation_results)
print(decoder_metrics)
decoder_validation_results.head(5)

# Save to CSV
#decoder_validation_results.to_csv("gpt2_results.csv", index=False)

decoder train: 9741 examples
decoder validation: 1221 examples


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Decoder evaluating: 100%|██████████| 120/120 [01:02<00:00,  1.92it/s]

{'n': 120, 'accuracy': 0.19166666666666668, 'avg_confidence': 0.6684292916227712, 'confidence_correctness_corr': -0.17159804464971332, 'ece': 0.4767626249561045}


,question,answer_key,predicted,correct,confidence,all_probs
0,The man took paperwork to other people to cons...,B,A,0,0.438374,"{'A': 0.43837431781872843, 'B': 0.062804011767..."
1,"If a person is working a lot, what are they li...",A,A,1,0.611078,"{'A': 0.6110781330478768, 'B': 0.0656460670720..."
2,Where can you put a picture frame when it's no...,E,A,0,0.978278,"{'A': 0.9782780951974175, 'B': 0.0177167041256..."
3,"He had no issue committing perjury, he had a w...",A,B,0,0.930228,"{'A': 0.0012782565984964603, 'B': 0.9302281851..."
4,The boat passenger was explaining his fear of ...,E,C,0,0.777023,"{'A': 0.1495762999491364, 'B': 0.0193148728609..."


## Model 2: Encoder model RoBERTa

This is an earlier approach, where I found the "common token bias" as identified by Zhao et al. (the tendency of language models to prefer tokens that are frequent in pre-training data). They show that this causes models to heavily favour certain labels regardless of the question. This corresponds with the A-bias problem observed. Their fix is to estimate the model's bias by feeding a content-free "N/A" input and then calibrating the output distribution to be uniform across answers. That principle is what the null-prompt subtraction later is based on.

In [ ]:
# Split train data since we use the original validation set as test
# df_train, df_new_val = train_test_split(df, test_size=0.1, random_state=42)
# print(f"Train: {len(df_train)} - New val: {len(df_new_val)} - Test: {len(df_val)}")

# Load the model 
tokenizer = AutoTokenizer.from_pretrained("roberta-base")
model = AutoModelForMaskedLM.from_pretrained("roberta-base")
model.eval()

# Shared base template (14-20 tokens w/ choices clearly delimited)
prompt_template = ("Question: {question}\n"
    "A) {A}\nB) {B}\nC) {C}\nD) {D}\nE) {E}\n"
    "Answer:")

# Prompt template for zero-shot prompting 
def build_prompt_roberta(question, choices):
    """
    Args:
        question (str): the question string
        choices (dict): {'A': 'text', 'B': 'text', ...}
    Returns:
        str: the full prompt with a <mask> token for RoBERTa to fill
    """
    return prompt_template.format(question=question, **choices) + " <mask>" 

# Define scoring function 
# Encoders (like RoBERTa) don't generate text, so instead we score each
# choice label (A-E) by how likely the model predicts it at the <mask> position.
# -> Confidence score = the softmax probability of each label token? 
def get_label_probabilities(prompt):
    """
    Returns a dict of {label: probability} for A–E at the <mask> position
    """
    inputs = tokenizer(prompt, return_tensors="pt")
    mask_index = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]

    with torch.no_grad():
        outputs = model(**inputs)

    logits_at_mask = outputs.logits[0, mask_index, :]  # shape: (1, vocab_size)

    # Get token IDs for labels A–E (with leading space, as RoBERTa tokenizes " A" not "A")
    label_token_ids = {
        label: tokenizer.convert_tokens_to_ids(f"Ġ{label}")  # Ġ = space prefix in RoBERTa vocab
        for label in ["A", "B", "C", "D", "E"]}

    # Softmax over just the 5 label tokens to get relative confidence
    label_logits = torch.tensor([logits_at_mask[0, tid] for tid in label_token_ids.values()])
    probs = torch.softmax(label_logits, dim=0)

    return {label: probs[i].item() for i, label in enumerate(label_token_ids)}

# Run on 1 row to test out
sample = df.iloc[5]

prompt = build_prompt_roberta(sample["question"], sample["choices_parsed"])
probs  = get_label_probabilities(prompt)

predicted = max(probs, key=probs.get)
correct   = sample["answerKey"]

print(f"Question : {sample['question']}")
print(f"Probs : {probs}")
print(f"Predicted: {predicted}, the right answer was: {correct} -> So, answer was {'correct' if predicted == correct else 'wrong'}")

df_testing = df[:870]
# Run on full dataset 

# results = []
# for _, row in tqdm(df_testing.iterrows(), total = len(df_testing)):
#     prompt = build_prompt_roberta(row["question"], row["choices_parsed"])
#     probs  = get_label_probabilities(prompt)
#     predicted = max(probs, key=probs.get)
#     results.append({
#         "question"   : row["question"],
#         "answerKey"  : row["answerKey"],
#         "predicted"  : predicted,
#         "correct"    : predicted == row["answerKey"],
#         "confidence" : max(probs.values()), #confidence = prob of chosen answer
#         "all_probs"  : probs})

# df_results = pd.DataFrame(results)
# print(f"Accuracy: {df_results['correct'].mean():.3f}")
# df_results.to_csv("roberta_results.csv", index=False)


### Calibrated PLL

In [ ]:
# Define pll score
def pll_score(prompt, answer_text):
    # Tokenize full prompt
    ids = tokenizer(prompt, return_tensors="pt")["input_ids"][0]
    # Tokenize answer 
    ans_ids = tokenizer(answer_text, add_special_tokens=False)["input_ids"]
    L = len(ans_ids)
    # Answer appears at the end of the prompt -> find its start index
    start = len(ids) - L

    total = 0.0
    # Compute pseudo-log-likelihood by masking each answer token once
    for i in range(start, start + L):
        masked = ids.clone()
        masked[i] = tokenizer.mask_token_id
        with torch.no_grad():
            logits = model(masked.unsqueeze(0)).logits[0, i]
        # Log-prob of the true token at this position
        log_probs = torch.log_softmax(logits, dim=-1)
        total += log_probs[ids[i]].item()

    # Length-normalized PLL
    return total / L


def score_calibrated(question, choices):
    scores = {}
    for label, text in choices.items():
        # Prompt w/ question (contextual likelihood)
        prompt_q = f"Question: {question}\nAnswer: {text}"
        # Null prompt -> estimates model's inherent bias toward the answer
        prompt_null = f"<mask> {text}"

        # Compute PLL for both
        score_q = pll_score(prompt_q, text)
        score_null = pll_score(prompt_null, text)

        # Subtract to remove token-frequency bias
        scores[label] = score_q - score_null

    return scores

In [ ]:
df_testing = df[:10]
results = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    question = row["question"]
    choices  = row["choices_parsed"]

    # Get calibrated PLL scores for each answer option
    scores = score_calibrated(question, choices)

    # Convert scores -> probabilities (softmax)
    labels = list(scores.keys())
    score_tensor = torch.tensor([scores[l] for l in labels])
    probs = torch.softmax(score_tensor, dim=0)

    # Pick the highest-prob answer
    pred_idx = torch.argmax(probs).item()
    predicted = labels[pred_idx]
    confidence = probs[pred_idx].item()

    # Save result
    results.append({
        "question"   : question,
        "answerKey"  : row["answerKey"],
        "predicted"  : predicted,
        "correct"    : predicted == row["answerKey"],
        "confidence" : confidence,
        "all_probs"  : {labels[i]: probs[i].item() for i in range(len(labels))},
        "all_scores" : scores})

# Convert to DataFrame
df_results = pd.DataFrame(results)

print(f"Accuracy (Calibrated PLL): {df_results['correct'].mean():.3f}")

# Save to CSV
df_results.to_csv("roberta_calibrated_pll_results_fulltrain.csv", index=False)

#On 500 rows: 0.264 aaccuracy in 9 minutes 